![Image Name](https://cdn.kesci.com/upload/image/rgg18bbfpr.png?imageView2/0/w/320/h/320)

# 材料二：基于深度学习的内容过滤

在这个练习中，你将使用神经网络实现基于内容的过滤，建立一个电影的推荐系统。


## 1 - 包
我们将使用一些熟悉的包，例如NumPy, TensorFlow 和[scikit-learn](https://scikit-learn.org/stable/)中的一些方法。
我们也会使用 [tabulate](https://pypi.org/project/tabulate/) 来更好地展现表格以及[Pandas](https://pandas.pydata.org/) 来管理表格数据。

In [2]:
import numpy as np
import numpy.ma as ma
from numpy import genfromtxt
from collections import defaultdict
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
import tabulate
from recsysNN_utils import *
from public_tests2 import test_sq_dist


pd.set_option("display.precision", 1)

## 2 - 电影评分数据集

这份数据集来自 [MovieLens ml-latest-small](https://grouplens.org/datasets/movielens/latest/) 数据集

[F. Maxwell Harper and Joseph A. Konstan. 2015. The MovieLens Datasets: History and Context. ACM Transactions on Interactive Intelligent Systems (TiiS) 5, 4: 19:1–19:19. <https://doi.org/10.1145/2827872>]

原始数据集有9000部电影，由600个用户进行评分，评分标准为0.5至5，以0.5为单位递增。该数据集的规模已经缩小，主要集中在2000年以后的电影和流行的类型。减少后的数据集有$n_u=395$用户和$n_m=694$电影。对于每部电影，数据集提供了一个电影名称、发行日期和一个或多个类型。例如，《玩具总动员3》于2010年上映，有几种类型："冒险|动画|儿童|喜剧|奇幻|IMAX"。 这个数据集除了他们的评分之外，几乎不包含任何关于用户的信息。这个数据集被用来为下面描述的神经网络创建训练向量。

### 2.1 用神经网络进行基于内容的过滤

在协同过滤实验中，你产生了两个向量，一个是用户向量，一个是项目/电影向量，这两个向量的点积可以预测一个评分。这些向量完全来自评分。  

基于内容的过滤也产生了一个用户和电影的特征向量，但认识到可能有关于用户和/或电影的其他信息，可以改善预测结果。这些额外的信息被提供给一个神经网络，然后生成用户和电影的向量，如下图所示。

![Image Name](https://cdn.kesci.com/upload/image/rgg21uqfgl.png?imageView2/0/w/960/h/960)

提供给网络的电影内容是原始数据和一些 "工程化特征 "的组合。回顾一下课程1第2周第4个实验中的特征工程讨论和实验。原始特征是电影上映的年份和电影的类型，以单点向量的形式呈现。有14种类型。设计的特征是来自用户评分的平均评分。具有多种类型的电影，每个类型都有一个训练向量。

用户内容只由工程特征组成。每个类型的平均评分是按用户计算的。此外，用户ID、评分计数和平均评分都是可用的，但不包括在训练或预测内容中。它们在解释数据时是很有用的。

训练集由数据集中的用户所做的所有评分组成。用户和电影/项目向量作为一个训练集一起提交给上述网络。用户向量对所有由用户评分的电影都是一样的。

下面，让我们加载并显示一些数据。

In [3]:
# Load Data, set configuration variables
item_train, user_train, y_train, item_features, user_features, item_vecs, movie_dict, user_to_genre = load_data()

num_user_features = user_train.shape[1] - 3  # remove userid, rating count and ave rating during training
num_item_features = item_train.shape[1] - 1  # remove movie id at train time
uvs = 3  # user genre vector start
ivs = 3  # item genre vector start
u_s = 3  # start of columns to use in training, user
i_s = 1  # start of columns to use in training, items
scaledata = True  # applies the standard scalar to data if true
print(f"Number of training vectors: {len(item_train)}")

Number of training vectors: 15574
Number of training vectors: 15574


一些用户和项目/电影的特征在训练中没有被使用。下面，括号中的特征"[]"，如 "用户ID"，"评分数 "和 "评分前 "在模型训练和使用时不包括在内。注意，用户向量对所有被评分的电影都是一样的。

In [4]:
pprint_train(user_train, user_features, uvs,  u_s, maxcount=5)

+-----------+----------------+--------------+--------+-----------+-----------+----------+--------+-------+-------------+-------+---------+--------+---------+---------+--------+----------+
| [user_id] | [rating_count] | [rating_ave] | Action | Adventure | Animation | Children | Comedy | Crime | Documentary | Drama | Fantasy | Horror | Mystery | Romance | Sci-Fi | Thriller |
+-----------+----------------+--------------+--------+-----------+-----------+----------+--------+-------+-------------+-------+---------+--------+---------+---------+--------+----------+
|    1.0    |      53.0      |     3.3      |  4.00  |   3.50    |   2.50    |   3.50   |  3.40  | 4.30  |    4.00     | 3.40  |  2.50   |  3.80  |  3.20   |  4.50   |  2.20  |   1.70   |
|    1.0    |      53.0      |     3.3      |  4.00  |   3.50    |   2.50    |   3.50   |  3.40  | 4.30  |    4.00     | 3.40  |  2.50   |  3.80  |  3.20   |  4.50   |  2.20  |   1.70   |
|    1.0    |      53.0      |     3.3      |  4.00  |   3.5

In [5]:
pprint_train(item_train, item_features, ivs, i_s, maxcount=5, user=False)

+------------+--------+------------+--------+-----------+-----------+----------+--------+-------+-------------+-------+---------+--------+---------+---------+--------+----------+
| [movie_id] |  year  | ave_rating | Action | Adventure | Animation | Children | Comedy | Crime | Documentary | Drama | Fantasy | Horror | Mystery | Romance | Sci-Fi | Thriller |
+------------+--------+------------+--------+-----------+-----------+----------+--------+-------+-------------+-------+---------+--------+---------+---------+--------+----------+
|   4974.0   | 2001.0 |    4.2     |        |           |           |          |  1.00  |       |             |       |         |        |         |         |        |          |
|   4032.0   | 2000.0 |    3.7     |        |           |           |          |  1.00  |       |             |       |         |        |         |         |        |          |
|   3324.0   | 2000.0 |    3.1     |        |           |           |          |  1.00  |       |        

In [6]:
print(f"y_train[:5]: {y_train[:5]}")

y_train[:5]: [3.5 3.5 3.5 3.5 3. ]


以上，我们可以看到，电影6874是一部在2003年上映的动作电影。用户2对动作电影的平均评分为3.9。此外，电影6874也被列在犯罪和惊悚片类型中。MovieLens的用户给这部电影的平均评分是4分。一个训练例子由两个表中的一行和y_train的评分组成。

### 2.2 准备训练数据
记得在课程1的第2周，我们探讨了将特征缩放作为提高收敛性的一种手段。我们将使用[scikit learn StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)对输入特征进行缩放。

这在课程1第2周的实验5中使用过。 下面还显示了反变换（inverse_transform）来产生原始输入。

In [7]:
# scale training data
if scaledata:
    item_train_save = item_train
    user_train_save = user_train

    scalerItem = StandardScaler()
    scalerItem.fit(item_train)
    item_train = scalerItem.transform(item_train)

    scalerUser = StandardScaler()
    scalerUser.fit(user_train)
    user_train = scalerUser.transform(user_train)

    print(np.allclose(item_train_save, scalerItem.inverse_transform(item_train)))
    print(np.allclose(user_train_save, scalerUser.inverse_transform(user_train)))

True
True


为了使我们能够评估结果，我们将把数据分成训练集和测试集，正如在课程2第3周讨论的那样。这里我们将使用[sklean train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)来分割和打乱数据。

请注意，将初始随机状态设置为相同的值，可以确保项目、用户和y的洗牌过程是相同的。

In [8]:
item_train, item_test = train_test_split(item_train, train_size=0.80, shuffle=True, random_state=1)
user_train, user_test = train_test_split(user_train, train_size=0.80, shuffle=True, random_state=1)
y_train, y_test       = train_test_split(y_train,    train_size=0.80, shuffle=True, random_state=1)
print(f"movie/item training data shape: {item_train.shape}")
print(f"movie/item test  data shape: {item_test.shape}")

movie/item training data shape: (12459, 17)
movie/item test  data shape: (3115, 17)


经过缩放、打乱的数据现在的平均值为零。

In [9]:
pprint_train(user_train, user_features, uvs, u_s, maxcount=5)

+-----------+----------------+--------------+--------+-----------+-----------+----------+--------+-------+-------------+-------+---------+--------+---------+---------+--------+----------+
| [user_id] | [rating_count] | [rating_ave] | Action | Adventure | Animation | Children | Comedy | Crime | Documentary | Drama | Fantasy | Horror | Mystery | Romance | Sci-Fi | Thriller |
+-----------+----------------+--------------+--------+-----------+-----------+----------+--------+-------+-------------+-------+---------+--------+---------+---------+--------+----------+
|    0.7    |      0.8       |     -0.4     |  1.56  |   -1.33   |   0.42    |   0.45   | -1.21  | -0.68 |    0.22     | -1.09 |  1.11   |  1.61  |  -1.07  |  -1.12  | -0.04  |   0.78   |
|    1.3    |      0.5       |     -0.8     | -0.85  |   -1.21   |   1.36    |  -1.05   |  1.30  | 0.14  |    1.25     | -0.15 |  1.58   | -1.09  |  0.23   |  -1.35  | -0.15  |   0.21   |
|    0.9    |      -1.4      |     1.1      |  0.48  |   -0.

使用Min Max Scaler来缩放目标评级，使其在-1和1之间。我们使用scikit-learn，因为它有一个反变换。[scikit learn MinMaxScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html)

In [10]:
scaler = MinMaxScaler((-1, 1))
scaler.fit(y_train.reshape(-1, 1))
ynorm_train = scaler.transform(y_train.reshape(-1, 1))
ynorm_test = scaler.transform(y_test.reshape(-1, 1))
print(ynorm_train.shape, ynorm_test.shape)

(12459, 1) (3115, 1)


## 3 - 基于内容过滤的神经网络


![Image Name](https://cdn.kesci.com/upload/image/rghffdckxb.png?imageView2/0/w/960/h/960)


现在，让我们构建一个如上图所述的神经网络。它将有两个网络，通过点积来组合。你将构建这两个网络。在这个例子中，它们将是相同的。请注意，这些网络不需要是相同的。如果用户内容比电影内容大得多，你可以选择增加用户网络相对于电影网络的复杂性。在这种情况下，内容是相似的，所以网络是相同的。

- 使用Keras的顺序模型
    - 第一层是一个密集层，有256个单元和一个relu激活。
    - 第二层是密集层，有128个单元和一个relu激活。
    - 第三层是一个密集层，有`num_outputs`单元和一个线性或无激活。  
    
该网络的其余部分将被提供。所提供的代码没有使用Keras顺序模型，而是使用Keras [functional api]（https://keras.io/guides/functional_api/）。

这种格式使得组件的相互连接方式更加灵活。


In [11]:
# GRADED_CELL
# UNQ_C1

num_outputs = 32
tf.random.set_seed(1)
user_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  tf.keras.layers.Dense(256, activation='relu'),
  tf.keras.layers.Dense(128, activation='relu'),
  tf.keras.layers.Dense(num_outputs),
    ### END CODE HERE ###  
])

item_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  tf.keras.layers.Dense(256, activation='relu'),
  tf.keras.layers.Dense(128, activation='relu'),
  tf.keras.layers.Dense(num_outputs),
    ### END CODE HERE ###  
])

# create the user input and point to the base network
input_user = tf.keras.layers.Input(shape=(num_user_features,))
vu = user_NN(input_user)
# vu = tf.linalg.l2_normalize(vu, axis=1)
vu = tf.keras.layers.BatchNormalization(axis=1)(vu)

# create the item input and point to the base network
input_item = tf.keras.layers.Input(shape=(num_item_features,))
vm = item_NN(input_item)
# vm = tf.linalg.l2_normalize(vm, axis=1)
vm = tf.keras.layers.BatchNormalization(axis=1)(vm)

# compute the dot product of the two vectors vu and vm
output = tf.keras.layers.Dot(axes=1)([vu, vm])
# output = tf.keras.layers.dot([vu,vm],axes=1)

# specify the inputs and output of the model
model = tf.keras.Model([input_user, input_item], output)

model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 14)                │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ input_layer_2 (InputLayer)    │ (None, 16)                │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ sequential (Sequential)       │ (None, 32)                │          40,864 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ sequential_1 (Sequential)     │ (None, 32)                │          41,376 │ input_layer_2[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization           │ (None, 32)                │             128 │ sequential[0][0]           │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_1         │ (None, 32)                │             128 │ sequential_1[0][0]         │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dot (Dot)                     │ (None, 1)                 │               0 │ batch_normalization[0][0], │
│                               │                           │                 │ batch_normalization_1[0][… │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 82,496 (322.25 KB)

 Trainable params: 82,368 (321.75 KB)

 Non-trainable params: 128 (512.00 B)

实现提示：

	你可以像下面展示的那样创建一个带有relu激活函数的稠密层
    
```python 
user_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  tf.keras.layers.Dense(256, activation='relu'),
  tf.keras.layers.Dense(128, activation='relu'),
  tf.keras.layers.Dense(num_outputs),
    ### END CODE HERE ###  
])

item_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  tf.keras.layers.Dense(256, activation='relu'),
  tf.keras.layers.Dense(128, activation='relu'),
  tf.keras.layers.Dense(num_outputs),
    ### END CODE HERE ###  
])
```

我们将使用均方误差损失和Adam优化

In [15]:
tf.random.set_seed(1)
cost_fn = tf.keras.losses.MeanSquaredError()
opt = keras.optimizers.Adam(learning_rate=0.01)
model.compile(optimizer=opt,
              loss=cost_fn)

In [16]:
model.fit([user_train[:, u_s:], item_train[:, i_s:]], ynorm_train, epochs=30)

Epoch 1/30
390/390 ━━━━━━━━━━━━━━━━━━━━ 2s 820us/step - loss: 1.6436
Epoch 2/30
390/390 ━━━━━━━━━━━━━━━━━━━━ 0s 779us/step - loss: 0.1072
Epoch 3/30
390/390 ━━━━━━━━━━━━━━━━━━━━ 0s 790us/step - loss: 0.0872
Epoch 4/30
390/390 ━━━━━━━━━━━━━━━━━━━━ 0s 786us/step - loss: 0.0814
Epoch 5/30
390/390 ━━━━━━━━━━━━━━━━━━━━ 0s 784us/step - loss: 0.0765
Epoch 6/30
390/390 ━━━━━━━━━━━━━━━━━━━━ 0s 789us/step - loss: 0.0710
Epoch 7/30
390/390 ━━━━━━━━━━━━━━━━━━━━ 0s 796us/step - loss: 0.0666
Epoch 8/30
390/390 ━━━━━━━━━━━━━━━━━━━━ 0s 785us/step - loss: 0.0642
Epoch 9/30
390/390 ━━━━━━━━━━━━━━━━━━━━ 0s 787us/step - loss: 0.0622
Epoch 10/30
390/390 ━━━━━━━━━━━━━━━━━━━━ 0s 784us/step - loss: 0.0603
Epoch 11/30
390/390 ━━━━━━━━━━━━━━━━━━━━ 0s 792us/step - loss: 0.0604
Epoch 12/30
390/390 ━━━━━━━━━━━━━━━━━━━━ 0s 802us/step - loss: 0.0646
Epoch 13/30
390/390 ━━━━━━━━━━━━━━━━━━━━ 0s 780us/step - loss: 0.0625
Epoch 14/30
390/390 ━━━━━━━━━━━━━━━━━━━━ 0s 786us/step - loss: 0.0563
Epoch 15/30
390/390 ━━━━━━━━━

评估模型以确定测试数据的损失。它与训练损失相当，表明该模型没有严重过拟合训练数据。

In [17]:
model.evaluate([user_test[:, u_s:], item_test[:, i_s:]], ynorm_test)

98/98 ━━━━━━━━━━━━━━━━━━━━ 0s 625us/step - loss: 0.0552


0.0551554411649704

### 3.1 预测
下面，你将使用你的模型在一些情况下进行预测。
#### 对一个新用户的预测
首先，我们将创建一个新用户，让模型为该用户推荐电影。在你试过这个例子的用户内容后，请随意改变用户内容以符合你自己的喜好，看看模型的建议。请注意，评分是在0.5和5.0之间，以0.5为单位递增。

In [21]:
new_user_id = 5000
new_rating_ave = 4.0
new_action = 1
new_adventure = 1
new_animation = 1
new_childrens = 1
new_comedy = 1
new_crime = 1
new_documentary = 5
new_drama = 1
new_fantasy = 1
new_horror = 1
new_mystery = 1
new_romance = 1
new_scifi = 5
new_thriller = 1
new_rating_count = 8

user_vec = np.array([[new_user_id, new_rating_count, new_rating_ave,
                      new_action, new_adventure, new_animation, new_childrens,
                      new_comedy, new_crime, new_documentary,
                      new_drama, new_fantasy, new_horror, new_mystery,
                      new_romance, new_scifi, new_thriller]])

让我们看一下新用户的最高评分电影。回顾一下，用户向量的类型倾向于喜剧和浪漫。
下面，我们将使用一组电影/项目向量，`item_vecs`，在训练/测试集中的每部电影都有一个向量。这与上面的用户向量相匹配，缩放后的向量被用来预测上述新用户的所有电影的评分。

In [22]:
# generate and replicate the user vector to match the number movies in the data set.
user_vecs = gen_user_vecs(user_vec,len(item_vecs))

# scale the vectors and make predictions for all movies. Return results sorted by rating.
sorted_index, sorted_ypu, sorted_items, sorted_user = predict_uservec(user_vecs,  item_vecs, model, u_s, i_s, 
                                                                       scaler, scalerUser, scalerItem, scaledata=scaledata)

print_pred_movies(sorted_ypu, sorted_user, sorted_items, movie_dict, maxcount = 10)

22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 

                           Top Movie Recommendations                            
+------+-------------+------+--------------------------------------------------------------------+------------------+
| Rank | Pred Rating | Year |                            Movie Title                             |      Genres      |
+------+-------------+------+--------------------------------------------------------------------+------------------+
|  1   |    7.88     | 2000 |                   Eyes of Tammy Faye, The (2000)                   |   Documentary    |
|  2   |    7.86     | 2000 | Endurance: Shackleton's Legendary Antarctic Expedition, The (2000) |   Documentary    |
|  3   |    7.84     | 2000 |                Long Night's Journey Into Day (2000)                |   Documentary    |
|  4   |    7.82     | 2000 |       Gleaners & I, The (Les glaneurs et la glaneuse) (2000)       |   Documentary    |
|  5   |    7.75     | 2000 |                  Micha

如果你确实在上面创建了一个用户，那么值得注意的是，网络被训练为预测一个用户评级，这个用户向量包括一整个用户流派评级。 如果没有具有类似评分集的用户，简单地提供一个单一流派的最大评分和其他流派的最小评分对网络可能没有意义。

#### 对一个现有用户的预测。
让我们看看对 "用户36 "的预测，这是数据集中的一个用户。我们可以将预测的评分与模型的评分进行比较。请注意，具有多种类型的电影在训练数据中出现了多次。例如，"The Time Machine "有三种类型。冒险、动作、科幻

In [23]:
uid =  36 
# form a set of user vectors. This is the same vector, transformed and repeated.
user_vecs, y_vecs = get_user_vecs(uid, scalerUser.inverse_transform(user_train), item_vecs, user_to_genre)

# scale the vectors and make predictions for all movies. Return results sorted by rating.
sorted_index, sorted_ypu, sorted_items, sorted_user = predict_uservec(user_vecs, item_vecs, model, u_s, i_s, scaler, 
                                                                      scalerUser, scalerItem, scaledata=scaledata)
sorted_y = y_vecs[sorted_index]

#print sorted predictions
print_existing_user(sorted_ypu, sorted_y.reshape(-1,1), sorted_user, sorted_items, item_features, ivs, uvs, movie_dict, maxcount = 10)

22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 

                              Predictions for Existing User                               
+------+------+--------+-------+------+-------------------------------------+-----------------+
| Rank | Pred | Actual | Diff  | Year |             Movie Title             |     Genres      |
+------+------+--------+-------+------+-------------------------------------+-----------------+
|  1   | 4.35 |  2.55  | +1.81 | 2002 |         Brown Sugar (2002)          |     Romance     |
|  2   | 4.27 |  4.49  | -0.22 | 2003 |      All the Real Girls (2003)      |  Drama|Romance  |
|  3   | 4.23 |  2.02  | +2.22 | 2000 |          Claim, The (2000)          | Romance|Western |
|  4   | 4.20 |  2.89  | +1.31 | 2002 |        Moonlight Mile (2002)        |  Drama|Romance  |
|  5   | 4.19 |  2.27  | +1.93 | 2002 | Talk to Her (Hable con Ella) (2002) |  Drama|Romance  |
|  6   | 4.18 |  2.12  | +2.05 | 2002 |          Hours, The (2002)          |  Drama|Romance  |
|  7

#### 寻找相似项目
上面的神经网络产生两个特征向量，一个是用户特征向量$v_u$，另一个是电影特征向量$v_m$。这些是32个条目向量，其值很难解释。然而，类似的项目会有类似的向量。这个信息可以用来做推荐。例如，如果一个用户对《玩具总动员3》评价很高，就可以通过选择具有类似电影特征向量的电影来推荐类似的电影。

相似性的衡量标准是两个向量之间的平方距离 $\mathbf{v_m^{(k)}}$ 和 $\mathbf{v_m^{(i)}}$ :
$$
\left\Vert \mathbf{v_m^{(k)}} - \mathbf{v_m^{(i)}}  \right\Vert^2 = \sum_{l=1}^{n}(v_{m_l}^{(k)} - v_{m_l}^{(i)})^2\tag{1}
$$

### 练习 1

写一个函数来计算平方距离

In [24]:
# GRADED_FUNCTION: sq_dist
# UNQ_C2
def sq_dist(a,b):
    """
    Returns the squared distance between two vectors
    Args:
      a (ndarray (n,)): vector with n features
      b (ndarray (n,)): vector with n features
    Returns:
      d (float) : distance
    """
    ### START CODE HERE ###     
    d = np.sum(np.square(a-b))
    ### END CODE HERE ###     
    return (d)

In [25]:
# Public tests
test_sq_dist(sq_dist)

Testing sq_dist function...
  Test 1 passed: sq_dist([1,2,3], [1,2,3]) = 0.0
  Test 2 passed: sq_dist([1.1,2.1,3.1], [1,2,3]) = 0.0300
  Test 3 passed: sq_dist([0,1,0], [1,0,0]) = 2
  Test 4 passed: sq_dist([1,2,3,4,5], [5,4,3,2,1]) = 40
  Test 5 passed: sq_dist([0.5,1.5,2.5], [1.5,2.5,3.5]) = 3.0

All tests passed!


True

In [26]:
a1 = np.array([1.0, 2.0, 3.0]); b1 = np.array([1.0, 2.0, 3.0])
a2 = np.array([1.1, 2.1, 3.1]); b2 = np.array([1.0, 2.0, 3.0])
a3 = np.array([0, 1, 0]);       b3 = np.array([1, 0, 0])
print(f"squared distance between a1 and b1: {sq_dist(a1, b1)}")
print(f"squared distance between a2 and b2: {sq_dist(a2, b2)}")
print(f"squared distance between a3 and b3: {sq_dist(a3, b3)}")

squared distance between a1 and b1: 0.0
squared distance between a2 and b2: 0.030000000000000054
squared distance between a3 and b3: 2


电影之间的距离矩阵可以在模型训练时计算一次，然后重新用于新的推荐，而无需重新训练。一旦模型训练完成，第一步是为每部电影获得电影特征向量，$v_m$。为了做到这一点，我们将使用训练好的`item_NN`并建立一个小模型，让我们通过它来运行电影向量，以生成$v_m$。

In [27]:
input_item_m = tf.keras.layers.Input(shape=(num_item_features,))    # input layer
vm_m = item_NN(input_item_m)                                       # use the trained item_NN
# vm_m = tf.linalg.l2_normalize(vm_m, axis=1)                        # incorporate normalization as was done in the original model
vm_m = tf.keras.layers.BatchNormalization(axis=1)(vm_m)
model_m = tf.keras.Model(input_item_m, vm_m)
model_m.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)           │ (None, 16)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ sequential_1 (Sequential)            │ (None, 32)                  │          41,376 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_2                │ (None, 32)                  │             128 │
│ (BatchNormalization)                 │                             │                 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 41,504 (162.12 KB)

 Trainable params: 41,440 (161.88 KB)

 Non-trainable params: 64 (256.00 B)

一旦你有了一个电影模型，你就可以通过使用模型来预测，使用一组项目/电影向量作为输入，创建一组电影特征向量。`item_vecs`是所有电影向量的集合。回顾一下，同一部电影将作为其每个类型的单独向量出现。它必须被缩放，以便与训练好的模型一起使用。预测的结果是每部电影的32条特征向量。

In [28]:
scaled_item_vecs = scalerItem.transform(item_vecs)
vms = model_m.predict(scaled_item_vecs[:,i_s:])
print(f"size of all predicted movie feature vectors: {vms.shape}")

22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
size of all predicted movie feature vectors: (694, 32)


现在我们来计算每个电影特征向量与所有其他电影特征向量之间的平方距离矩阵。

![Image Name](https://cdn.kesci.com/upload/image/rghpcqhmiy.PNG?imageView2/0/w/960/h/960)


然后，我们可以通过沿着每一行找到最小值来找到最接近的电影。我们将利用[numpy masked arrays](https://numpy.org/doc/1.21/user/tutorial-ma.html)来避免选择相同的电影。沿着对角线的屏蔽值不会被包括在计算中。

In [29]:
count = 50
dim = len(vms)
dist = np.zeros((dim,dim))

for i in range(dim):
    for j in range(dim):
        dist[i,j] = sq_dist(vms[i, :], vms[j, :])
        
m_dist = ma.masked_array(dist, mask=np.identity(dist.shape[0]))  # mask the diagonal

disp = [["movie1", "genres", "movie2", "genres"]]
for i in range(count):
    min_idx = np.argmin(m_dist[i])
    movie1_id = int(item_vecs[i,0])
    movie2_id = int(item_vecs[min_idx,0])
    genre1,_  = get_item_genre(item_vecs[i,:], ivs, item_features)
    genre2,_  = get_item_genre(item_vecs[min_idx,:], ivs, item_features)

    disp.append( [movie_dict[movie1_id]['title'], genre1,
                  movie_dict[movie2_id]['title'], genre2]
               )
table = tabulate.tabulate(disp, tablefmt='html', headers="firstrow", floatfmt=[".1f", ".1f", ".0f", ".2f", ".2f"])
table

movie1,genres,movie2,genres
"Yards, The (2000)",Crime|Drama,"Man Who Wasn't There, The (2001)",Crime|Drama
Next Friday (2000),Comedy,I'm the One That I Want (2000),Comedy
Supernova (2000),Adventure|Sci-Fi|Thriller,Spider-Man (2002),Action|Adventure|Sci-Fi|Thriller
Down to You (2000),Comedy|Romance,Boys and Girls (2000),Comedy|Romance
Scream 3 (2000),Comedy|Horror|Mystery|Thriller,"Ring, The (2002)",Horror|Mystery|Thriller
"Boondock Saints, The (2000)",Action|Crime|Drama|Thriller,Spy Game (2001),Action|Crime|Drama|Thriller
Gun Shy (2000),Comedy,Snow Day (2000),Comedy
"Beach, The (2000)",Adventure|Drama,Gerry (2002),Adventure|Drama
Snow Day (2000),Comedy,Gun Shy (2000),Comedy
"Tigger Movie, The (2000)",Animation|Children,"Road to El Dorado, The (2000)",Animation|Children


结果显示，该模型将推荐一部来自同一类型的电影。

## 4 - 祝贺你! 

![Image Name](https://cdn.kesci.com/upload/image/rghpf1no4h.png?imageView2/0/w/320/h/320)

你已经完成了一个基于内容的推荐系统。   

这种结构是许多商业推荐系统的基础。用户内容可以大大扩展，以纳入更多关于用户的信息，如果这些信息可用的话。 项目不限于电影。这可以用来推荐任何物品、书籍、汽车或与你 "购物车 "中的物品相似的物品。